# Web y APIs · FastAPI

APIs con `fastapi`, que valida los datos a partir de las anotaciones de tipos y de modelos de `pydantic` y genera la documentación OpenAPI automáticamente. En un proyecto real la aplicación vive en su propio fichero y se arranca con `fastapi dev main.py` (o `uvicorn main:app`); aquí cada aplicación se escribe en un módulo con `%%writefile` y se prueba con `TestClient` de `fastapi.testclient`, que hace las peticiones sin levantar un servidor.

## 1. Primera aplicación

Escribir una aplicación con una ruta `GET /` que devuelva un mensaje en JSON y otra `GET /saluda/{nombre}` que salude a la persona indicada. Probarla con `TestClient` y comprobar qué responde una ruta que no existe.

In [1]:
%%writefile hola.py
from fastapi import FastAPI

app = FastAPI(title="Hola")


@app.get("/")
def inicio():
    return {"mensaje": "Hola, FastAPI"}


@app.get("/saluda/{nombre}")
def saluda(nombre: str):
    return {"saludo": f"Hola, {nombre}"}

Writing hola.py


In [2]:
import warnings

# Starlette avisa de que su cliente de pruebas pasará de httpx a httpx2; no afecta a los ejercicios
warnings.filterwarnings("ignore", message="Using `httpx` with `starlette.testclient`")
from fastapi.testclient import TestClient

import hola

cliente = TestClient(hola.app)
r = cliente.get("/")
print(r.status_code, r.headers["content-type"], r.json())
print(cliente.get("/saluda/Ana").json())
r = cliente.get("/no-existe")
print(r.status_code, r.json())
assert cliente.get("/saluda/Luis").json() == {"saludo": "Hola, Luis"}

200 application/json {'mensaje': 'Hola, FastAPI'}
{'saludo': 'Hola, Ana'}
404 {'detail': 'Not Found'}


## 2. Parámetros de ruta y de consulta

Crear una calculadora con `GET /cuadrado/{n}` (parámetro de ruta entero), `GET /divisores/{n}` que solo acepte $1 \le n \le 10\,000$ con `Path` y `GET /potencia?base=...&exponente=...` (parámetros de consulta, con exponente 2 por defecto). Comprobar que FastAPI convierte los tipos y responde con 422 cuando un valor no es válido o falta.

In [3]:
%%writefile calculadora.py
from typing import Annotated

from fastapi import FastAPI, Path

app = FastAPI(title="Calculadora")


@app.get("/cuadrado/{n}")
def cuadrado(n: int):
    return {"n": n, "cuadrado": n * n}


@app.get("/divisores/{n}")
def divisores(n: Annotated[int, Path(ge=1, le=10_000)]):
    return [d for d in range(1, n + 1) if n % d == 0]


@app.get("/potencia")
def potencia(base: float, exponente: int = 2):
    return {"resultado": base**exponente}

Writing calculadora.py


In [4]:
import calculadora

cliente = TestClient(calculadora.app)
print(cliente.get("/cuadrado/7").json(), cliente.get("/divisores/28").json())
print(cliente.get("/potencia", params={"base": 2, "exponente": 10}).json(), cliente.get("/potencia?base=1.5").json())
for url in ("/cuadrado/siete", "/divisores/0", "/potencia"):
    r = cliente.get(url)
    error = r.json()["detail"][0]
    print(r.status_code, url, error["loc"], error["msg"])
assert cliente.get("/divisores/12").json() == [1, 2, 3, 4, 6, 12]

{'n': 7, 'cuadrado': 49} [1, 2, 4, 7, 14, 28]
{'resultado': 1024.0} {'resultado': 2.25}
422 /cuadrado/siete ['path', 'n'] Input should be a valid integer, unable to parse string as an integer
422 /divisores/0 ['path', 'n'] Input should be greater than or equal to 1
422 /potencia ['query', 'base'] Field required


## 3. Cuerpo de la petición con modelos Pydantic

Crear `POST /pedidos/total`, que reciba en JSON un pedido con el cliente, una lista no vacía de líneas (producto, precio positivo y unidades, 1 por defecto) y un cupón opcional, y devuelva el número de artículos y el total (con un 10 % de descuento con el cupón `DESCUENTO10`). Enviar un pedido válido y otro con varios errores y mostrar los errores de validación (422).

In [5]:
%%writefile pedidos.py
from fastapi import FastAPI
from pydantic import BaseModel, Field

app = FastAPI(title="Pedidos")


class Linea(BaseModel):
    producto: str = Field(min_length=1)
    precio: float = Field(gt=0)
    unidades: int = Field(default=1, ge=1)


class Pedido(BaseModel):
    cliente: str
    lineas: list[Linea] = Field(min_length=1)
    cupon: str | None = None


@app.post("/pedidos/total")
def total(pedido: Pedido):
    importe = sum(linea.precio * linea.unidades for linea in pedido.lineas)
    if pedido.cupon == "DESCUENTO10":
        importe *= 0.9
    return {"cliente": pedido.cliente, "articulos": sum(linea.unidades for linea in pedido.lineas), "total": round(importe, 2)}

Writing pedidos.py


In [6]:
import pedidos

cliente = TestClient(pedidos.app)
valido = {"cliente": "Ana", "lineas": [{"producto": "teclado", "precio": 49.5, "unidades": "2"}, {"producto": "ratón", "precio": 19.9}]}
r = cliente.post("/pedidos/total", json=valido)
print(r.status_code, r.json())
print(cliente.post("/pedidos/total", json={**valido, "cupon": "DESCUENTO10"}).json())

r = cliente.post("/pedidos/total", json={"lineas": [{"producto": "", "precio": -3}]})
print(r.status_code)
for error in r.json()["detail"]:
    print(error["loc"], error["msg"])
assert r.status_code == 422 and len(r.json()["detail"]) == 3

200 {'cliente': 'Ana', 'articulos': 3, 'total': 118.9}
{'cliente': 'Ana', 'articulos': 3, 'total': 107.01}
422
['body', 'cliente'] Field required
['body', 'lineas', 0, 'producto'] String should have at least 1 character
['body', 'lineas', 0, 'precio'] Input should be greater than 0


## 4. Modelo de respuesta

Crear `POST /usuarios` para registrar usuarios con nombre de usuario (3 a 20 minúsculas, dígitos o `_`), correo y clave de al menos 8 caracteres. La respuesta debe tener el código 201 y no incluir nunca la clave: usar un modelo de salida distinto en `response_model`.

In [7]:
%%writefile usuarios.py
from fastapi import FastAPI, status
from pydantic import BaseModel, Field

app = FastAPI(title="Usuarios")


class UsuarioEntrada(BaseModel):
    usuario: str = Field(pattern=r"^[a-z0-9_]{3,20}$")
    email: str = Field(pattern=r"^[\w.+-]+@[\w-]+\.[\w.]+$")
    clave: str = Field(min_length=8)


class UsuarioSalida(BaseModel):
    usuario: str
    email: str


@app.post("/usuarios", response_model=UsuarioSalida, status_code=status.HTTP_201_CREATED)
def registra(datos: UsuarioEntrada):
    return datos  # response_model descarta la clave

Writing usuarios.py


In [8]:
import usuarios

cliente = TestClient(usuarios.app)
r = cliente.post("/usuarios", json={"usuario": "ana_garcia", "email": "ana@correo.es", "clave": "una clave larga"})
print(r.status_code, r.json())
assert r.status_code == 201 and "clave" not in r.json()
r = cliente.post("/usuarios", json={"usuario": "Ana García", "email": "ana@correo", "clave": "corta"})
print(r.status_code, [error["loc"][-1] for error in r.json()["detail"]])

201 {'usuario': 'ana_garcia', 'email': 'ana@correo.es'}
422 ['usuario', 'email', 'clave']


## 5. Agenda de contactos (CRUD)

Escribir una API para una agenda guardada en memoria con las operaciones `GET /contactos` (con búsqueda opcional por nombre), `GET /contactos/{id}`, `POST /contactos`, `PUT /contactos/{id}` y `DELETE /contactos/{id}`. Usar los códigos de estado adecuados: 201 al crear, 204 al borrar, 404 si el contacto no existe y 409 si ya hay un contacto con el mismo teléfono, lanzando `HTTPException`.

In [9]:
%%writefile agenda.py
from fastapi import FastAPI, HTTPException, status
from pydantic import BaseModel, Field

app = FastAPI(title="Agenda", version="1.0")


class ContactoNuevo(BaseModel):
    nombre: str = Field(min_length=1)
    telefono: str = Field(pattern=r"^\+?\d{9,15}$")
    email: str | None = None


class Contacto(ContactoNuevo):
    id: int


contactos: dict[int, Contacto] = {}
siguiente_id = 1


def busca(id: int) -> Contacto:
    if id not in contactos:
        raise HTTPException(status.HTTP_404_NOT_FOUND, f"No existe el contacto {id}")
    return contactos[id]


def comprueba_telefono(telefono: str, id: int | None = None):
    if any(c.telefono == telefono and c.id != id for c in contactos.values()):
        raise HTTPException(status.HTTP_409_CONFLICT, "Ya hay un contacto con ese teléfono")


@app.get("/contactos")
def lista(q: str = "") -> list[Contacto]:
    return [c for c in contactos.values() if q.lower() in c.nombre.lower()]


@app.get("/contactos/{id}")
def lee(id: int) -> Contacto:
    return busca(id)


@app.post("/contactos", status_code=status.HTTP_201_CREATED)
def crea(datos: ContactoNuevo) -> Contacto:
    global siguiente_id
    comprueba_telefono(datos.telefono)
    contacto = Contacto(id=siguiente_id, **datos.model_dump())
    contactos[contacto.id] = contacto
    siguiente_id += 1
    return contacto


@app.put("/contactos/{id}")
def actualiza(id: int, datos: ContactoNuevo) -> Contacto:
    busca(id)
    comprueba_telefono(datos.telefono, id)
    contactos[id] = Contacto(id=id, **datos.model_dump())
    return contactos[id]


@app.delete("/contactos/{id}", status_code=status.HTTP_204_NO_CONTENT)
def borra(id: int) -> None:
    busca(id)
    del contactos[id]

Writing agenda.py


In [10]:
import agenda

cliente = TestClient(agenda.app)

def muestra(metodo, url, **kwargs):
    r = cliente.request(metodo, url, **kwargs)
    print(f"{metodo:7}{url:22}{r.status_code}  {r.text}")
    return r

muestra("POST", "/contactos", json={"nombre": "Ana García", "telefono": "+34600111222", "email": "ana@correo.es"})
muestra("POST", "/contactos", json={"nombre": "Luis Pérez", "telefono": "600333444"})
muestra("POST", "/contactos", json={"nombre": "Otra Ana", "telefono": "+34600111222"})
muestra("GET", "/contactos?q=ana")
muestra("PUT", "/contactos/2", json={"nombre": "Luis Pérez Gil", "telefono": "600333444"})
muestra("GET", "/contactos/2")
muestra("DELETE", "/contactos/1")
muestra("GET", "/contactos/1")
muestra("DELETE", "/contactos/1")
assert [c["nombre"] for c in cliente.get("/contactos").json()] == ["Luis Pérez Gil"]

POST   /contactos            201  {"nombre":"Ana García","telefono":"+34600111222","email":"ana@correo.es","id":1}
POST   /contactos            201  {"nombre":"Luis Pérez","telefono":"600333444","email":null,"id":2}
POST   /contactos            409  {"detail":"Ya hay un contacto con ese teléfono"}
GET    /contactos?q=ana      200  [{"nombre":"Ana García","telefono":"+34600111222","email":"ana@correo.es","id":1}]
PUT    /contactos/2          200  {"nombre":"Luis Pérez Gil","telefono":"600333444","email":null,"id":2}
GET    /contactos/2          200  {"nombre":"Luis Pérez Gil","telefono":"600333444","email":null,"id":2}
DELETE /contactos/1          204  
GET    /contactos/1          404  {"detail":"No existe el contacto 1"}
DELETE /contactos/1          404  {"detail":"No existe el contacto 1"}


## 6. Manejador de errores propio

En una API de cuentas bancarias, `POST /cuentas/{titular}/retirar?cantidad=...` resta dinero del saldo. Definir una excepción `SaldoInsuficiente` y registrar con `@app.exception_handler` una función que la convierta en una respuesta JSON con código 400, el saldo disponible y la cantidad pedida. Una cuenta inexistente debe responder 404 y una cantidad no positiva 422.

In [11]:
%%writefile banco.py
from typing import Annotated

from fastapi import FastAPI, HTTPException, Query, Request, status
from fastapi.responses import JSONResponse

app = FastAPI(title="Banco")
saldos = {"ana": 100.0, "luis": 20.0}


class SaldoInsuficiente(Exception):
    def __init__(self, saldo: float, cantidad: float):
        self.saldo, self.cantidad = saldo, cantidad


@app.exception_handler(SaldoInsuficiente)
async def saldo_insuficiente(request: Request, exc: SaldoInsuficiente):
    return JSONResponse(status_code=status.HTTP_400_BAD_REQUEST,
                        content={"error": "Saldo insuficiente", "saldo": exc.saldo, "cantidad": exc.cantidad})


@app.post("/cuentas/{titular}/retirar")
def retira(titular: str, cantidad: Annotated[float, Query(gt=0)]):
    if titular not in saldos:
        raise HTTPException(status.HTTP_404_NOT_FOUND, "Cuenta no encontrada")
    if cantidad > saldos[titular]:
        raise SaldoInsuficiente(saldos[titular], cantidad)
    saldos[titular] -= cantidad
    return {"titular": titular, "saldo": saldos[titular]}

Writing banco.py


In [12]:
import banco

cliente = TestClient(banco.app)
for titular, cantidad in [("ana", 30), ("ana", 80), ("luis", 20), ("pepe", 5), ("ana", -1)]:
    r = cliente.post(f"/cuentas/{titular}/retirar", params={"cantidad": cantidad})
    print(titular, cantidad, r.status_code, r.json())
assert banco.saldos == {"ana": 70.0, "luis": 0.0}

ana 30 200 {'titular': 'ana', 'saldo': 70.0}
ana 80 400 {'error': 'Saldo insuficiente', 'saldo': 70.0, 'cantidad': 80.0}
luis 20 200 {'titular': 'luis', 'saldo': 0.0}
pepe 5 404 {'detail': 'Cuenta no encontrada'}


ana -1 422 {'detail': [{'type': 'greater_than', 'loc': ['query', 'cantidad'], 'msg': 'Input should be greater than 0', 'input': '-1', 'ctx': {'gt': 0.0}}]}


## 7. Dependencias para autenticar con un token

Proteger una API con una dependencia `usuario_actual` que lea la cabecera `Authorization: Bearer <token>`, devuelva el usuario asociado al token y responda 401 si falta o no es válido. Añadir otra dependencia reutilizable para paginar (`saltar` y `limite`). Probar la ruta pública, las protegidas con y sin token, y sustituir la dependencia en las pruebas con `app.dependency_overrides`.

In [13]:
%%writefile protegida.py
from typing import Annotated

from fastapi import Depends, FastAPI, Header, HTTPException, Query, status

app = FastAPI(title="API protegida")
TOKENS = {"token-de-ana": "ana", "token-de-luis": "luis"}


def usuario_actual(authorization: Annotated[str | None, Header()] = None) -> str:
    tipo, _, token = (authorization or "").partition(" ")
    if tipo != "Bearer" or token not in TOKENS:
        raise HTTPException(status.HTTP_401_UNAUTHORIZED, "Token ausente o no válido", headers={"WWW-Authenticate": "Bearer"})
    return TOKENS[token]


def paginacion(saltar: Annotated[int, Query(ge=0)] = 0, limite: Annotated[int, Query(ge=1, le=50)] = 10) -> slice:
    return slice(saltar, saltar + limite)


Usuario = Annotated[str, Depends(usuario_actual)]
Pagina = Annotated[slice, Depends(paginacion)]


@app.get("/publica")
def publica():
    return {"mensaje": "Cualquiera puede verlo"}


@app.get("/perfil")
def perfil(usuario: Usuario):
    return {"usuario": usuario}


@app.get("/notas")
def notas(usuario: Usuario, pagina: Pagina):
    return [f"Nota {i} de {usuario}" for i in range(1, 26)][pagina]

Writing protegida.py


In [14]:
import protegida

cliente = TestClient(protegida.app)
print(cliente.get("/publica").json())
r = cliente.get("/perfil")
print(r.status_code, r.json(), r.headers["www-authenticate"])
print(cliente.get("/perfil", headers={"Authorization": "Bearer falso"}).status_code)
ana = {"Authorization": "Bearer token-de-ana"}
print(cliente.get("/perfil", headers=ana).json())
print(cliente.get("/notas", headers=ana, params={"saltar": 20, "limite": 3}).json())
print(cliente.get("/notas", headers=ana, params={"limite": 100}).status_code)

protegida.app.dependency_overrides[protegida.usuario_actual] = lambda: "pruebas"
print(cliente.get("/perfil").json())
protegida.app.dependency_overrides.clear()
assert cliente.get("/perfil").status_code == 401

{'mensaje': 'Cualquiera puede verlo'}
401 {'detail': 'Token ausente o no válido'} Bearer
401
{'usuario': 'ana'}
['Nota 21 de ana', 'Nota 22 de ana', 'Nota 23 de ana']
422


{'usuario': 'pruebas'}


## 8. Rutas asíncronas

Una ruta `async def` puede esperar operaciones de entrada y salida (otra API, una base de datos) con `await` sin bloquear el servidor, que mientras tanto atiende otras peticiones. Simular tres consultas lentas a otros servicios con `asyncio.sleep(0.2)` y comparar una ruta que las espera una tras otra con otra que las lanza a la vez con `asyncio.gather`.

In [15]:
%%writefile asincrona.py
import asyncio
import time

from fastapi import FastAPI

app = FastAPI(title="Asíncrona")
SERVICIOS = ("tiempo", "tráfico", "noticias")


async def consulta(servicio: str) -> str:
    await asyncio.sleep(0.2)
    return f"datos de {servicio}"


@app.get("/secuencial")
async def secuencial():
    inicio = time.perf_counter()
    datos = [await consulta(s) for s in SERVICIOS]
    return {"datos": datos, "segundos": round(time.perf_counter() - inicio, 2)}


@app.get("/concurrente")
async def concurrente():
    inicio = time.perf_counter()
    datos = await asyncio.gather(*(consulta(s) for s in SERVICIOS))
    return {"datos": datos, "segundos": round(time.perf_counter() - inicio, 2)}

Writing asincrona.py


In [16]:
import asincrona

cliente = TestClient(asincrona.app)
secuencial, concurrente = cliente.get("/secuencial").json(), cliente.get("/concurrente").json()
print(secuencial)
print(concurrente)
assert secuencial["datos"] == concurrente["datos"]
assert secuencial["segundos"] >= 0.6 and concurrente["segundos"] < 0.4

{'datos': ['datos de tiempo', 'datos de tráfico', 'datos de noticias'], 'segundos': 0.62}
{'datos': ['datos de tiempo', 'datos de tráfico', 'datos de noticias'], 'segundos': 0.21}


## 9. Documentación OpenAPI

FastAPI describe la API en `/openapi.json` y la muestra de forma interactiva en `/docs` (Swagger UI) y `/redoc`. Obtener el esquema de la agenda, listar cada ruta con su método, su resumen y sus códigos de respuesta, y mostrar los campos obligatorios y el patrón del teléfono del modelo `ContactoNuevo`.

In [17]:
cliente = TestClient(agenda.app)
esquema = cliente.get("/openapi.json").json()
print(esquema["openapi"], esquema["info"])
for ruta, operaciones in esquema["paths"].items():
    for metodo, operacion in operaciones.items():
        print(f"{metodo.upper():7}{ruta:18}{operacion['summary']:12}{sorted(operacion['responses'])}")
modelo = esquema["components"]["schemas"]["ContactoNuevo"]
print(modelo["required"], modelo["properties"]["telefono"])
print(cliente.get("/docs").status_code, cliente.get("/redoc").status_code)
assert set(esquema["paths"]) == {"/contactos", "/contactos/{id}"}

3.1.0 {'title': 'Agenda', 'version': '1.0'}
GET    /contactos        Lista       ['200', '422']
POST   /contactos        Crea        ['201', '422']
GET    /contactos/{id}   Lee         ['200', '422']
PUT    /contactos/{id}   Actualiza   ['200', '422']
DELETE /contactos/{id}   Borra       ['204', '422']
['nombre', 'telefono'] {'type': 'string', 'pattern': '^\\+?\\d{9,15}$', 'title': 'Telefono'}
200 200
